# 📋 Notebook 1 — Data Loading & EDA
**Dataset:** `speeches_modeling.csv` (~46,000 Dutch parliamentary speeches)

Key columns used:
- `speech_text` — full transcript text → input to RobBERT
- `speaker_name` / `speaker_party` — politician & party
- `motion_passed` — binary target (1=passed, 0=rejected)
- `hour_of_day`, `time_bin`, `hour_sin`, `hour_cos`, `is_weekend` — already engineered
- `speech_duration_seconds` — speech length in seconds
- `is_voorzitter` — whether speaker is the chair

Run order: NB1 → NB2 → NB3 → NB4 → NB5


In [ ]:
# !pip install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import re, warnings
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi':130,'axes.spines.top':False,'axes.spines.right':False})
BLUE,RED,GREEN,ORANGE = '#2B5797','#C0392B','#27AE60','#E67E22'
print("Libraries loaded ✓")

## 1. Load & Inspect

In [ ]:
DATA_PATH = "speeches_modeling.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Shape: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")

In [ ]:
print("=== dtypes ===")
print(df.dtypes.to_string())
print("\n=== Null counts (non-zero only) ===")
nulls = df.isnull().sum()
print(nulls[nulls > 0].to_string())

In [ ]:
# Preview real speech content
sample = df[['speaker_name','speaker_party','speech_text','motion_passed']].dropna(subset=['speech_text']).sample(3, random_state=42)
for _, row in sample.iterrows():
    print(f"Speaker : {row['speaker_name']}  ({row['speaker_party']})")
    print(f"Passed  : {row['motion_passed']}")
    print(f"Text    : {str(row['speech_text'])[:300]}")
    print()

## 2. Speech Text Length

In [ ]:
# Drop rows with no speech text
before = len(df)
df = df.dropna(subset=['speech_text']).copy()
df = df[df['speech_text'].str.strip().str.len() > 10].copy()
print(f"Dropped {before - len(df)} rows with missing/empty speech_text | Remaining: {len(df):,}")

df['text_chars']    = df['speech_text'].str.len()
df['approx_tokens'] = (df['text_chars'] / 4).astype(int)  # ~4 chars per Dutch token

print("\n=== Character length stats ===")
print(df['text_chars'].describe().round(1).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Character distribution
ax = axes[0]
ax.hist(df['text_chars'].clip(upper=3000), bins=80, color=BLUE, edgecolor='white', alpha=0.85)
ROBBERT_LIMIT = 512 * 4  # ~2048 chars
ax.axvline(ROBBERT_LIMIT, color=RED, linestyle='--', lw=1.5, label=f'RobBERT ~512-token limit (~{ROBBERT_LIMIT} chars)')
for pct, color, ls in [(50, GREEN, ':'),(75, ORANGE, ':'),(95, RED, ':')]:
    val = df['text_chars'].quantile(pct/100)
    ax.axvline(val, color=color, linestyle=ls, lw=1.2, label=f'p{pct} = {val:.0f} chars')
ax.set_xlabel('Speech Length (characters)'); ax.set_ylabel('Number of Speeches')
ax.set_title('Distribution of Speech Text Length (characters)', fontweight='bold')
ax.legend(fontsize=8)

# Token distribution
ax2 = axes[1]
ax2.hist(df['approx_tokens'].clip(upper=1000), bins=80, color=ORANGE, edgecolor='white', alpha=0.85)
ax2.axvline(512, color=RED, linestyle='--', lw=1.5, label='RobBERT max = 512 tokens')
pct_over = (df['approx_tokens'] > 512).mean() * 100
ax2.set_xlabel('Estimated RobBERT Tokens'); ax2.set_ylabel('Number of Speeches')
ax2.set_title('Estimated RobBERT Token Count per Speech', fontweight='bold')
ax2.legend(fontsize=8)
ax2.text(520, ax2.get_ylim()[1]*0.5, f'{pct_over:.1f}%\nexceed\n512 tokens', color=RED, fontsize=9)

plt.tight_layout(); plt.show()
print(f"\nSpeeches likely exceeding 512 tokens: {pct_over:.1f}%  → will be chunked in NB3")

## 3. Target Variable

In [ ]:
df['motion_passed'] = pd.to_numeric(df['motion_passed'], errors='coerce')
df_lab = df.dropna(subset=['motion_passed']).copy()
df_lab['motion_passed'] = df_lab['motion_passed'].astype(int)

n_total = len(df_lab)
n_pass  = df_lab['motion_passed'].sum()
n_rej   = n_total - n_pass
ratio   = max(n_pass, n_rej) / min(n_pass, n_rej)

print(f"Total labelled speeches: {n_total:,}")
print(f"  Passed   (1): {n_pass:,}  ({n_pass/n_total*100:.1f}%)")
print(f"  Rejected (0): {n_rej:,}  ({n_rej/n_total*100:.1f}%)")
print(f"  Class ratio:  {ratio:.2f}:1")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].pie([n_pass, n_rej], labels=['Passed','Rejected'], colors=[GREEN, RED],
            autopct='%1.1f%%', startangle=90, wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[0].set_title('Motion Outcome Distribution', fontweight='bold')

if 'motion_outcome_raw' in df_lab.columns:
    rc = df_lab['motion_outcome_raw'].value_counts().head(10)
    axes[1].barh(rc.index, rc.values, color=BLUE, edgecolor='white', alpha=0.85)
    axes[1].set_title('Top Raw Outcome Labels', fontweight='bold')

plt.tight_layout(); plt.show()

## 4. Speaker & Party Analysis

In [ ]:
top_spk = (df_lab.groupby('speaker_name')
             .agg(n_speeches=('speech_id','count'), pass_rate=('motion_passed','mean'))
             .sort_values('n_speeches', ascending=False).head(20))

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].barh(top_spk.index[::-1], top_spk['n_speeches'][::-1], color=BLUE, edgecolor='white', alpha=0.85)
axes[0].set_title('Top 20 Speakers — Speech Count', fontweight='bold')
axes[0].set_xlabel('Number of Speeches')

cols_s = [GREEN if v >= 0.5 else RED for v in top_spk['pass_rate'][::-1]]
axes[1].barh(top_spk.index[::-1], top_spk['pass_rate'][::-1]*100, color=cols_s, edgecolor='white', alpha=0.85)
axes[1].axvline(50, color='grey', linestyle='--', lw=0.8)
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_title('Top 20 Speakers — Motion Pass Rate', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
party = (df_lab.groupby('speaker_party')
           .agg(n=('speech_id','count'), pass_rate=('motion_passed','mean'))
           .sort_values('n', ascending=False).head(15))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(party.index, party['n'], color=BLUE, edgecolor='white', alpha=0.85)
axes[0].set_title('Speeches per Party (top 15)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

cols_p = [GREEN if v >= 0.5 else RED for v in party['pass_rate']]
axes[1].bar(party.index, party['pass_rate']*100, color=cols_p, edgecolor='white', alpha=0.85)
axes[1].axhline(50, color='grey', linestyle='--')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_title('Pass Rate per Party', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
n_voorzitter = (df_lab['is_voorzitter'].astype(str).str.lower().isin(['true','1'])).sum()
print(f"Voorzitter (chair) speeches: {n_voorzitter:,} ({n_voorzitter/len(df_lab)*100:.1f}%)")
print("→ Flagged with is_voorzitter feature — chair speeches contain less opinion/sentiment")

## 5. Time-of-Day Patterns

In [ ]:
tod = df_lab.groupby('time_bin')['motion_passed'].agg(['mean','count']).rename(columns={'mean':'pass_rate','count':'n'})

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors_t = [GREEN if v >= 0.5 else RED for v in tod['pass_rate']]
bars = axes[0].bar(tod.index, tod['pass_rate']*100, color=colors_t, edgecolor='white', alpha=0.85, width=0.6)
axes[0].axhline(50, color='grey', linestyle='--', lw=0.8)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].set_title('Pass Rate by Time of Day', fontweight='bold'); axes[0].set_ylim(0,80)
axes[0].tick_params(axis='x', rotation=25)
for bar, n in zip(bars, tod['n']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'n={n}', ha='center', fontsize=7.5, color='#555')

hr = df_lab[df_lab['hour_of_day']>=0].groupby('hour_of_day')['motion_passed'].mean()
axes[1].bar(hr.index, hr.values*100, color=BLUE, edgecolor='white', alpha=0.8)
axes[1].axhline(50, color=RED, linestyle='--', lw=0.8)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_xlabel('Hour of Day'); axes[1].set_title('Pass Rate by Hour', fontweight='bold')
axes[1].set_xticks(range(0,24,2))
plt.tight_layout(); plt.show()

In [ ]:
df_hm = df_lab[(df_lab['hour_of_day']>=0) & df_lab['day_of_week'].notna()]
hm = df_hm.groupby(['hour_of_day','day_of_week'])['motion_passed'].mean().unstack(fill_value=np.nan)
hm.columns = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun'][:len(hm.columns)]

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(hm, ax=ax, cmap='RdYlGn', vmin=0, vmax=1, linewidths=0.3, linecolor='white',
            annot=True, fmt='.2f', cbar_kws={'label':'Pass Rate','format':'%.0%%'})
ax.set_title('Pass Rate: Hour of Day × Day of Week', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 6. Save Clean Dataset

In [ ]:
df_out = df_lab.rename(columns={'motion_passed':'label','hour_of_day':'hour'}).reset_index(drop=True)
df_out.to_csv("speeches_clean.csv", index=False)
print(f"Saved {len(df_out):,} rows → speeches_clean.csv")